In [ ]:
"""
Analyse des Facteurs de Performance des Élèves
Dataset Kaggle: Données sur les facteurs de performance des élèves
Auteur: Assimedi Akhate
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, mean_squared_error, r2_score)
from sklearn.tree import DecisionTreeClassifier
import warnings
warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*70)
print("ANALYSE DES FACTEURS DE PERFORMANCE DES ÉLÈVES")
print("="*70)

# ============================================================================
# PARTIE 1: CHARGEMENT ET EXPLORATION DES DONNÉES
# ============================================================================

# Option 1: Si vous avez téléchargé le dataset depuis Kaggle
# Remplacez le chemin par votre fichier local
csv_path = 'student_performance_factors.csv'

try:
    df = pd.read_csv(csv_path)
    print(f"\n✓ Dataset chargé avec succès!")
    print(f"  Dimensions: {df.shape[0]} lignes × {df.shape[1]} colonnes\n")
except FileNotFoundError:
    print(f"\n✗ Fichier non trouvé: {csv_path}")
    print("  Instructions:")
    print("  1. Téléchargez le dataset depuis Kaggle")
    print("  2. Placez le fichier CSV dans le même répertoire")
    print("  3. Modifiez 'csv_path' si nécessaire\n")
    exit()

# Aperçu des données
print("📊 APERÇU DES PREMIÈRES LIGNES:")
print(df.head())

print("\n📋 INFORMATIONS SUR LES COLONNES:")
print(df.info())

print("\n📈 STATISTIQUES DESCRIPTIVES:")
print(df.describe())

print("\n🔍 VALEURS MANQUANTES:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("  Aucune valeur manquante détectée!")

# ============================================================================
# PARTIE 2: NETTOYAGE ET PRÉPARATION DES DONNÉES
# ============================================================================

print("\n" + "="*70)
print("PRÉPARATION DES DONNÉES")
print("="*70)

# Copie du dataframe original
df_clean = df.copy()

# Gestion des valeurs manquantes
for col in df_clean.columns:
    if df_clean[col].dtype == 'object':
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
    else:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

print(f"✓ Valeurs manquantes traitées")

# Encodage des variables catégorielles
le = LabelEncoder()
categorical_cols = df_clean.select_dtypes(include=['object']).columns

print(f"\n📝 Variables catégorielles identifiées: {len(categorical_cols)}")
for col in categorical_cols:
    df_clean[col + '_encoded'] = le.fit_transform(df_clean[col])
    print(f"  - {col}: {df_clean[col].nunique()} catégories")

# ============================================================================
# PARTIE 3: ANALYSE EXPLORATOIRE DES DONNÉES (EDA)
# ============================================================================

print("\n" + "="*70)
print("ANALYSE EXPLORATOIRE")
print("="*70)

# Corrélation entre les variables numériques
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
correlation_matrix = df_clean[numeric_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1)
plt.title('Matrice de Corrélation - Facteurs de Performance',
          fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
print("\n✓ Matrice de corrélation sauvegardée: correlation_matrix.png")

# Distribution de la variable cible (à adapter selon votre dataset)
# Exemple: si 'Exam_Score' ou 'Final_Grade' est la variable cible
target_candidates = ['Exam_Score', 'ExamScore', 'Final_Grade', 'FinalGrade',
                     'Performance', 'Grade', 'Score']
target_col = None

for candidate in target_candidates:
    if candidate in df_clean.columns:
        target_col = candidate
        break

if target_col:
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    df_clean[target_col].hist(bins=30, edgecolor='black')
    plt.title(f'Distribution de {target_col}', fontweight='bold')
    plt.xlabel(target_col)
    plt.ylabel('Fréquence')

    plt.subplot(1, 2, 2)
    df_clean[target_col].plot(kind='box')
    plt.title(f'Boîte à moustaches - {target_col}', fontweight='bold')
    plt.ylabel(target_col)

    plt.tight_layout()
    plt.savefig('target_distribution.png', dpi=300, bbox_inches='tight')
    print(f"✓ Distribution de {target_col} sauvegardée: target_distribution.png")

# ============================================================================
# PARTIE 4: MODÉLISATION MACHINE LEARNING
# ============================================================================

print("\n" + "="*70)
print("MODÉLISATION MACHINE LEARNING")
print("="*70)

if target_col:
    # Préparation des features et de la cible
    feature_cols = [col for col in df_clean.select_dtypes(include=[np.number]).columns
                    if col != target_col and 'encoded' not in col]

    X = df_clean[feature_cols]
    y = df_clean[target_col]

    # Division train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Normalisation
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"\n📊 Données d'entraînement: {X_train.shape[0]} exemples")
    print(f"📊 Données de test: {X_test.shape[0]} exemples")
    print(f"📊 Nombre de features: {X_train.shape[1]}")

    # Déterminer si c'est un problème de régression ou classification
    is_classification = y.nunique() < 20

    if is_classification:
        print("\n🎯 Type de problème: CLASSIFICATION")
        print(f"   Classes: {y.nunique()}")

        # Random Forest Classifier
        print("\n🌲 Random Forest Classifier")
        rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        rf_clf.fit(X_train_scaled, y_train)
        y_pred_rf = rf_clf.predict(X_test_scaled)

        accuracy = accuracy_score(y_test, y_pred_rf)
        print(f"   Précision: {accuracy:.4f} ({accuracy*100:.2f}%)")

        print("\n📊 Rapport de Classification:")
        print(classification_report(y_test, y_pred_rf))

        # Importance des features
        feature_importance = pd.DataFrame({
            'feature': feature_cols,
            'importance': rf_clf.feature_importances_
        }).sort_values('importance', ascending=False)

        print("\n🔝 TOP 10 FACTEURS LES PLUS IMPORTANTS:")
        print(feature_importance.head(10).to_string(index=False))

        # Visualisation importance des features
        plt.figure(figsize=(12, 6))
        sns.barplot(data=feature_importance.head(10), x='importance', y='feature')
        plt.title('Top 10 Facteurs de Performance', fontsize=14, fontweight='bold')
        plt.xlabel('Importance')
        plt.tight_layout()
        plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
        print("\n✓ Graphique d'importance sauvegardé: feature_importance.png")

    else:
        print("\n🎯 Type de problème: RÉGRESSION")

        # Random Forest Regressor
        print("\n🌲 Random Forest Regressor")
        rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf_reg.fit(X_train_scaled, y_train)
        y_pred_rf = rf_reg.predict(X_test_scaled)

        mse = mean_squared_error(y_test, y_pred_rf)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred_rf)

        print(f"   RMSE: {rmse:.4f}")
        print(f"   R² Score: {r2:.4f} ({r2*100:.2f}%)")

        # Importance des features
        feature_importance = pd.DataFrame({
            'feature': feature_cols,
            'importance': rf_reg.feature_importances_
        }).sort_values('importance', ascending=False)

        print("\n🔝 TOP 10 FACTEURS LES PLUS IMPORTANTS:")
        print(feature_importance.head(10).to_string(index=False))

        # Visualisation importance des features
        plt.figure(figsize=(12, 6))
        sns.barplot(data=feature_importance.head(10), x='importance', y='feature')
        plt.title('Top 10 Facteurs de Performance', fontsize=14, fontweight='bold')
        plt.xlabel('Importance')
        plt.tight_layout()
        plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
        print("\n✓ Graphique d'importance sauvegardé: feature_importance.png")

        # Graphique prédictions vs réalité
        plt.figure(figsize=(10, 6))
        plt.scatter(y_test, y_pred_rf, alpha=0.5)
        plt.plot([y_test.min(), y_test.max()],
                 [y_test.min(), y_test.max()],
                 'r--', lw=2)
        plt.xlabel('Valeurs Réelles')
        plt.ylabel('Valeurs Prédites')
        plt.title('Prédictions vs Réalité', fontweight='bold')
        plt.tight_layout()
        plt.savefig('predictions_vs_actual.png', dpi=300, bbox_inches='tight')
        print("✓ Graphique prédictions sauvegardé: predictions_vs_actual.png")

# ============================================================================
# PARTIE 5: EXPORT DES RÉSULTATS
# ============================================================================

print("\n" + "="*70)
print("EXPORT DES RÉSULTATS")
print("="*70)

# Sauvegarder le dataset nettoyé
df_clean.to_csv('student_performance_cleaned.csv', index=False)
print("✓ Dataset nettoyé: student_performance_cleaned.csv")

# Sauvegarder l'importance des features
if 'feature_importance' in locals():
    feature_importance.to_csv('feature_importance.csv', index=False)
    print("✓ Importance des features: feature_importance.csv")

print("\n" + "="*70)
print("ANALYSE TERMINÉE AVEC SUCCÈS!")
print("="*70)
print("\n📁 Fichiers générés:")
print("  - correlation_matrix.png")
print("  - target_distribution.png")
print("  - feature_importance.png")
if not is_classification:
    print("  - predictions_vs_actual.png")
print("  - student_performance_cleaned.csv")
print("  - feature_importance.csv")
print("\n💡 Utilisez ces résultats pour identifier les facteurs clés")
print("   de performance et améliorer les stratégies pédagogiques!")"""
Analyse des Facteurs de Performance des Élèves
Dataset Kaggle: Données sur les facteurs de performance des élèves
Auteur: Assimedi Akhate
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, mean_squared_error, r2_score)
from sklearn.tree import DecisionTreeClassifier
import warnings
warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*70)
print("ANALYSE DES FACTEURS DE PERFORMANCE DES ÉLÈVES")
print("="*70)

# ============================================================================
# PARTIE 1: CHARGEMENT ET EXPLORATION DES DONNÉES
# ============================================================================

# Option 1: Si vous avez téléchargé le dataset depuis Kaggle
# Remplacez le chemin par votre fichier local
csv_path = 'student_performance_factors.csv'

try:
    df = pd.read_csv(csv_path)
    print(f"\n✓ Dataset chargé avec succès!")
    print(f"  Dimensions: {df.shape[0]} lignes × {df.shape[1]} colonnes\n")
except FileNotFoundError:
    print(f"\n✗ Fichier non trouvé: {csv_path}")
    print("  Instructions:")
    print("  1. Téléchargez le dataset depuis Kaggle")
    print("  2. Placez le fichier CSV dans le même répertoire")
    print("  3. Modifiez 'csv_path' si nécessaire\n")
    exit()

# Aperçu des données
print("📊 APERÇU DES PREMIÈRES LIGNES:")
print(df.head())

print("\n📋 INFORMATIONS SUR LES COLONNES:")
print(df.info())

print("\n📈 STATISTIQUES DESCRIPTIVES:")
print(df.describe())

print("\n🔍 VALEURS MANQUANTES:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("  Aucune valeur manquante détectée!")

# ============================================================================
# PARTIE 2: NETTOYAGE ET PRÉPARATION DES DONNÉES
# ============================================================================

print("\n" + "="*70)
print("PRÉPARATION DES DONNÉES")
print("="*70)

# Copie du dataframe original
df_clean = df.copy()

# Gestion des valeurs manquantes
for col in df_clean.columns:
    if df_clean[col].dtype == 'object':
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
    else:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

print(f"✓ Valeurs manquantes traitées")

# Encodage des variables catégorielles
le = LabelEncoder()
categorical_cols = df_clean.select_dtypes(include=['object']).columns

print(f"\n📝 Variables catégorielles identifiées: {len(categorical_cols)}")
for col in categorical_cols:
    df_clean[col + '_encoded'] = le.fit_transform(df_clean[col])
    print(f"  - {col}: {df_clean[col].nunique()} catégories")

# ============================================================================
# PARTIE 3: ANALYSE EXPLORATOIRE DES DONNÉES (EDA)
# ============================================================================

print("\n" + "="*70)
print("ANALYSE EXPLORATOIRE")
print("="*70)

# Corrélation entre les variables numériques
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
correlation_matrix = df_clean[numeric_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1)
plt.title('Matrice de Corrélation - Facteurs de Performance',
          fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
print("\n✓ Matrice de corrélation sauvegardée: correlation_matrix.png")

# Distribution de la variable cible (à adapter selon votre dataset)
# Exemple: si 'Exam_Score' ou 'Final_Grade' est la variable cible
target_candidates = ['Exam_Score', 'ExamScore', 'Final_Grade', 'FinalGrade',
                     'Performance', 'Grade', 'Score']
target_col = None

for candidate in target_candidates:
    if candidate in df_clean.columns:
        target_col = candidate
        break

if target_col:
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    df_clean[target_col].hist(bins=30, edgecolor='black')
    plt.title(f'Distribution de {target_col}', fontweight='bold')
    plt.xlabel(target_col)
    plt.ylabel('Fréquence')

    plt.subplot(1, 2, 2)
    df_clean[target_col].plot(kind='box')
    plt.title(f'Boîte à moustaches - {target_col}', fontweight='bold')
    plt.ylabel(target_col)

    plt.tight_layout()
    plt.savefig('target_distribution.png', dpi=300, bbox_inches='tight')
    print(f"✓ Distribution de {target_col} sauvegardée: target_distribution.png")

# ============================================================================
# PARTIE 4: MODÉLISATION MACHINE LEARNING
# ============================================================================

print("\n" + "="*70)
print("MODÉLISATION MACHINE LEARNING")
print("="*70)

if target_col:
    # Préparation des features et de la cible
    feature_cols = [col for col in df_clean.select_dtypes(include=[np.number]).columns
                    if col != target_col and 'encoded' not in col]

    X = df_clean[feature_cols]
    y = df_clean[target_col]

    # Division train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Normalisation
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"\n📊 Données d'entraînement: {X_train.shape[0]} exemples")
    print(f"📊 Données de test: {X_test.shape[0]} exemples")
    print(f"📊 Nombre de features: {X_train.shape[1]}")

    # Déterminer si c'est un problème de régression ou classification
    is_classification = y.nunique() < 20

    if is_classification:
        print("\n🎯 Type de problème: CLASSIFICATION")
        print(f"   Classes: {y.nunique()}")

        # Random Forest Classifier
        print("\n🌲 Random Forest Classifier")
        rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        rf_clf.fit(X_train_scaled, y_train)
        y_pred_rf = rf_clf.predict(X_test_scaled)

        accuracy = accuracy_score(y_test, y_pred_rf)
        print(f"   Précision: {accuracy:.4f} ({accuracy*100:.2f}%)")

        print("\n📊 Rapport de Classification:")
        print(classification_report(y_test, y_pred_rf))

        # Importance des features
        feature_importance = pd.DataFrame({
            'feature': feature_cols,
            'importance': rf_clf.feature_importances_
        }).sort_values('importance', ascending=False)

        print("\n🔝 TOP 10 FACTEURS LES PLUS IMPORTANTS:")
        print(feature_importance.head(10).to_string(index=False))

        # Visualisation importance des features
        plt.figure(figsize=(12, 6))
        sns.barplot(data=feature_importance.head(10), x='importance', y='feature')
        plt.title('Top 10 Facteurs de Performance', fontsize=14, fontweight='bold')
        plt.xlabel('Importance')
        plt.tight_layout()
        plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
        print("\n✓ Graphique d'importance sauvegardé: feature_importance.png")

    else:
        print("\n🎯 Type de problème: RÉGRESSION")

        # Random Forest Regressor
        print("\n🌲 Random Forest Regressor")
        rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf_reg.fit(X_train_scaled, y_train)
        y_pred_rf = rf_reg.predict(X_test_scaled)

        mse = mean_squared_error(y_test, y_pred_rf)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred_rf)

        print(f"   RMSE: {rmse:.4f}")
        print(f"   R² Score: {r2:.4f} ({r2*100:.2f}%)")

        # Importance des features
        feature_importance = pd.DataFrame({
            'feature': feature_cols,
            'importance': rf_reg.feature_importances_
        }).sort_values('importance', ascending=False)

        print("\n🔝 TOP 10 FACTEURS LES PLUS IMPORTANTS:")
        print(feature_importance.head(10).to_string(index=False))

        # Visualisation importance des features
        plt.figure(figsize=(12, 6))
        sns.barplot(data=feature_importance.head(10), x='importance', y='feature')
        plt.title('Top 10 Facteurs de Performance', fontsize=14, fontweight='bold')
        plt.xlabel('Importance')
        plt.tight_layout()
        plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
        print("\n✓ Graphique d'importance sauvegardé: feature_importance.png")

        # Graphique prédictions vs réalité
        plt.figure(figsize=(10, 6))
        plt.scatter(y_test, y_pred_rf, alpha=0.5)
        plt.plot([y_test.min(), y_test.max()],
                 [y_test.min(), y_test.max()],
                 'r--', lw=2)
        plt.xlabel('Valeurs Réelles')
        plt.ylabel('Valeurs Prédites')
        plt.title('Prédictions vs Réalité', fontweight='bold')
        plt.tight_layout()
        plt.savefig('predictions_vs_actual.png', dpi=300, bbox_inches='tight')
        print("✓ Graphique prédictions sauvegardé: predictions_vs_actual.png")

# ============================================================================
# PARTIE 5: EXPORT DES RÉSULTATS
# ============================================================================

print("\n" + "="*70)
print("EXPORT DES RÉSULTATS")
print("="*70)

# Sauvegarder le dataset nettoyé
df_clean.to_csv('student_performance_cleaned.csv', index=False)
print("✓ Dataset nettoyé: student_performance_cleaned.csv")

# Sauvegarder l'importance des features
if 'feature_importance' in locals():
    feature_importance.to_csv('feature_importance.csv', index=False)
    print("✓ Importance des features: feature_importance.csv")

print("\n" + "="*70)
print("ANALYSE TERMINÉE AVEC SUCCÈS!")
print("="*70)
print("\n📁 Fichiers générés:")
print("  - correlation_matrix.png")
print("  - target_distribution.png")
print("  - feature_importance.png")
if not is_classification:
    print("  - predictions_vs_actual.png")
print("  - student_performance_cleaned.csv")
print("  - feature_importance.csv")
print("\n💡 Utilisez ces résultats pour identifier les facteurs clés")
print("   de performance et améliorer les stratégies pédagogiques!")